# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tenuka-R/FlyRank-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

My lane for the ML task will be ranking pages based on priority to rewrite.
This is because a ranking would be informative and show in order a list of pages whereas for classification or clustering it would only be able to show whether it could benefit from a rewrite or not, which is less informative than a ranking.

In [31]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

queue = pd.read_csv("outputs/refresh_queue_sample.csv")
queue[["final_rank", "content_id", "final_refresh_score", "suggested_action", "trend_direction"]].head(10)

,final_rank,content_id,final_refresh_score,suggested_action,trend_direction
0,1,content_1f080331fa2b,81.636697,refresh_and_review_ctr,down
1,2,content_6aa43079fb0c,81.447656,refresh_and_review_ctr,down
2,3,content_d6570c51c9bd,81.430346,refresh_and_review_ctr,down
3,4,content_72e800a9c214,81.034960,refresh_and_review_ctr,down
4,5,content_e04eb9549989,80.873188,refresh_and_review_ctr,down
5,6,content_b69288c5e701,80.754770,refresh_and_review_ctr,down
6,7,content_9b6df29f7889,80.632923,refresh_and_review_ctr,down
7,8,content_bb6ebb5ec8c8,80.371236,refresh_and_review_ctr,down
8,9,content_4d76cdb3387b,80.362748,refresh_and_review_ctr,down
9,10,content_b4f35d640b1c,80.321757,refresh,down


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

The label comes from a defined rule. The label is_declining_label is based on the trend direction. I did not design the rule myself but I know that it is rule based, not a observed outcome



In [26]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(df["trend_direction"].value_counts())
print("Positive rate (declining):", df["is_declining_label"].mean())
df[["content_id", "trend_direction", "trend_pct", "is_declining_label"]].head(10)

trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64
Positive rate (declining): 0.5420666666666667


,content_id,trend_direction,trend_pct,is_declining_label
0,content_304f48230142,down,-41.4,1
1,content_a1fb4e703a9e,down,-57.7,1
2,content_9aa793d4d895,down,-60.9,1
3,content_331d6c4de07b,stable,-13.8,0
4,content_d99b7a2d90ca,down,-34.7,1
5,content_d4084a4bc775,down,-38.9,1
6,content_9a34b442b552,down,-92.3,1
7,content_a63219c6e95a,stable,0.6,0
8,content_5e6c160719bc,down,-58.8,1
9,content_c27558df2b0c,down,-29.2,1


## 3. Success metric

*One metric you can defend. What number means 'good'?*

My metric would be precision@50 as it is a number that shows out of the top 50 pages that my ranking displays, how many of those would actually benefit from being rewritten. A number close to 1 would be a good number such as 0.8.

In [22]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import json

res = json.load(open("outputs/model_results.json"))
base = res["baseline"]["baseline_precision_at_50"]
rf   = res["models"]["random_forest"]["precision_at_50"]
print(f"Baseline rule    Precision@50: {base:.3f}   (~{round(base*50)} of top 50 right)")
print(f"Random forest    Precision@50: {rf:.3f}   (~{round(rf*50)} of top 50 right)")
print(f"Lift: {rf/base:.1f}x")
print("Validation split used:", res["split_strategy"])

Baseline rule    Precision@50: 0.240   (~12 of top 50 right)
Random forest    Precision@50: 0.740   (~37 of top 50 right)
Lift: 3.1x
Validation split used: client_holdout


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

One row refers to one page

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df.shape[0], "rows,", df.shape[1], "columns")
df.head(3)


30000 rows, 44 columns


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule or simple if-statement would not be very practiacl because signals that saeem to predict staleness may not actually make sense. An example would be that a page's search volume does not tell you whether the page actually gets traffic or not.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
corr = df["search_volume"].corr(df["impressions_90d"])
print(f"Correlation between search_volume and impressions_90d: {corr:.3f}")
print("Near zero -> keyword search volume barely predicts the traffic a page actually gets.")


Correlation between search_volume and impressions_90d: 0.001
Near zero -> keyword search volume barely predicts the traffic a page actually gets.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.